# Assembly of nanopore sequences

Jacobo de la Cuesta-Zuluaga, August 2026.

The aim of this notebook is to execute the `nf-core` pipeline `bacass` for the assembly of a bacterial
genome sequenced using nanopore. You can find the pipeline documentation
[here](https://nf-co.re/bacass/2.6.0).

The input is the `fastq.gz` file produced in notebook 01. The output is an assembled, polished and
annotated genome.


## Before we start

Run notebook 01 first and wait until the base calling job has finished. You should have a `fastq.gz`
file in `data/fastq_files`.

Besides the `VScode` environment, this notebook needs the `Nextflow` environment. 

Create it by running the following command on the terminal:

`conda env create -f envs/Nextflow.yaml`

You only need to create the environment once.

## What you'll need to change

| Variable | Where | What to put there |
|---|---|---|
| `base_dir` | Load libraries and set paths | The same one you used in notebook 01 |
| `repo_dir` | Load libraries and set paths | Where you cloned this repository |
| `k2_db` | Load libraries and set paths | Only if you are not on M3 |

Sample IDs are now taken from the file names, so the samples table doesn't need editing.

## Load libraries and set paths

In [1]:
library(tidyverse)
library(conflicted)

── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors


In [2]:
conflicts_prefer(dplyr::filter())

[conflicted] Will prefer dplyr::filter over any other package.


If you are assembling genomes from the reads generated in notebook 01, you should use
the same directories. `base_dir` has to be exactly the one you used there, otherwise the
`fastq` files won't be found.

In [3]:
# Directories
# Base directory
base_dir = "/PATH/TO/YOUR/FOLDER"

# Where you cloned the Huequito repository. Not necessarily inside base_dir
repo_dir = "/mnt/lustre/groups/maier/maide581/projects/Huequito"

# Data
data_dir = file.path(base_dir, "data")

# fastq files
fastq_dir = file.path(data_dir, "fastq_files")

# sheets dir
sheets_dir = file.path(data_dir, "sheets")
dir.create(sheets_dir) 

# assembly dir
assembly_dir = file.path(data_dir, "assembly")
dir.create(assembly_dir) 

# Nextflow intermediate files. Kept out of the results folder, since they
# take up a lot of space and can be deleted once the run is done
work_dir = file.path(data_dir, "nextflow_work")
dir.create(work_dir)

# Databases
# Used by Kraken2 to check that the reads are not contaminated with another organism
k2_db = "/mnt/lustre/groups/maier/databases/Kraken_Bracken/k2_standard_16gb/k2_standard_16gb_20240605.tar.gz"

# Software
conda_env = "Nextflow"

Warning message:
In dir.create(sheets_dir) :
  cannot create dir '/PATH/TO/YOUR/FOLDER/data/sheets', reason 'No such file or directory'
Warning message:
In dir.create(assembly_dir) :
  cannot create dir '/PATH/TO/YOUR/FOLDER/data/assembly', reason 'No such file or directory'
Warning message:
In dir.create(work_dir) :
  cannot create dir '/PATH/TO/YOUR/FOLDER/data/nextflow_work', reason 'No such file or directory'


In [4]:
# Check that the base directory exists
stopifnot(dir.exists(base_dir))

: [1m[33mError[39m:[22m
[33m![39m dir.exists(base_dir) is not TRUE

**Note** that `Huequito`'s repository includes a nextflow configuration file that increases the
baseline computational resources used by the pipeline. If you want to use the default resource
allocation, remove the `-c` argument from the `bacass` command. In most cases you won't need to change
anything, this is just for your information.

In [ ]:
# Custom config file
nextflow_config = file.path(repo_dir, "config/nextflow.config")

## Prepare tables

`nf-core` pipelines require you to provide a table with the path of each sample.
You could do this manually, but it is better to have some code help you with that.

The chunk below lists all the `fastq.gz` files in the sequences folder and creates a table with the
necessary columns.

**Note** that there a multiple columns with `NA`. This is because the assembly 
pipeline can use multiple read types as input. In this case, we will only use `LongFastQ` 

We'll use the file name as `ID`.
The `ID` column is taken from the file name, so the table works as is for any number of samples.
Anything other than letters, numbers and underscores is replaced, since the `ID` becomes the name of
an output folder.

In [ ]:
# List files and only retain fastq files
long_reads = list.files(fastq_dir, full.names = TRUE, pattern = "fastq.gz$")

# Stop if notebook 01 has not produced any fastq file yet
stopifnot(length(long_reads) > 0)

In [ ]:
# Create sample sheets
# Create samples table
samples_table <- data.frame(LongFastQ = long_reads) |>
    mutate(
        ID = str_remove(basename(LongFastQ), fixed(".fastq.gz")) |>
            str_replace_all("[^A-Za-z0-9_]", "_"),
        R1 = NA,
        R2 = NA,
        Fast5 = NA,
        GenomeSize = NA
    ) |>
    relocate(ID) |>
    relocate(LongFastQ, .after = R2)

samples_table

             ID R1 R2                                                                                                         LongFastQ Fast5 GenomeSize
1 MMC234_202311 NA NA /mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data/fastq_files/MMC234_202311.fastq.gz    NA         NA

Check the `ID` column before continuing.

The chunk below saves the sample sheet as a tab-separated file, which will be used as input for the
assembly pipeline.

In [ ]:
# Write file
bacass_samplesfile = file.path(sheets_dir, "Example_bacass_samples.tsv")

samples_table %>%
    write_tsv(bacass_samplesfile)

## Execute pipeline

The code below constructs the bash command to activate the conda environment, change to the assembly
directory, and run the `bacass` pipeline with all required arguments and resources.

What the arguments do:

- `--assembler unicycler`: puts the reads together into contigs
- `--assembly_type long`: we only have nanopore reads, no Illumina
- `--polish_method medaka`: corrects the remaining errors in the draft assembly
- `--annotation_tool prokka`: finds and labels the genes
- `--kraken2db`: identifies the species in the reads, to spot contamination
- `--skip_kmerfinder`: skips a second species identification step we don't need
- `-c` and `-profile`: resources and cluster settings, see above
- `-work-dir`: directory where the pipeline intermediate files will be stored

In [ ]:
# Create command

bacass_cmd <- str_glue(
  "conda activate {{conda_env}} && \\
  cd {{out_dir}} && \\
  nextflow run nf-core/bacass -r 2.6.0 \\
  -profile m3c \\
  --input {{samples_sheet}} \\
  --outdir {{assemblies_dir}} \\
  -work-dir {{work_dir}} \\
  -c {{nextflow_config}} \\
  --kraken2db {{kraken_db}} \\
  --annotation_tool prokka \\
  --assembler unicycler \\
  --assembly_type long \\
  --polish_method medaka \\
  --skip_kmerfinder \\
  --skip_toulligqc"
)

Now we can replace the placeholders in the bacass command template with the actual paths and filenames
defined above. Then, the chunk prints the full command for you to copy and run in your terminal.

In [ ]:
# Fill template
assembly_cmd = str_glue(bacass_cmd,
                        conda_env = conda_env,
                        out_dir = assembly_dir,
                        samples_sheet = bacass_samplesfile,
                        assemblies_dir = assembly_dir,
                        kraken_db = k2_db)[]

assembly_cmd

conda activate Nextflow && cd /mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data/assembly && nextflow run nf-core/bacass -r 2.6.0 -profile m3c --input /mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data/sheets/Example_bacass_samples.tsv --outdir /mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data/assembly -work-dir /mnt/lustre/groups/maier/maide581/projects/Small_projects/Huequito_remake/data/nextflow_work -c /mnt/lustre/groups/maier/maide581/projects/Huequito/config/nextflow.config --kraken2db /mnt/lustre/groups/maier/databases/Kraken_Bracken/k2_standard_16gb/k2_standard_16gb_20240605.tar.gz --annotation_tool prokka --assembler unicycler --assembly_type long --polish_method medaka --skip_kmerfinder  --skip_toulligqc

## While it runs

The pipeline takes several hours and stops if your connection to the cluster drops. Start it inside a
`tmux` or `screen` session so it keeps running when you close the terminal.

Nextflow prints one line per step. If a step fails, adding `-resume` to the command restarts the run
from that point instead of from the beginning.

## What you should have at the end

Inside `data/assembly`:

- `medaka/`: the assembled and polished genome as a `fasta` file
- `Prokka/`: the annotation, including the `.gff` and `.faa` files
- `kraken2/`: which species the reads were assigned to
- `multiqc/multiqc_report.html`: the summary to open first

For a clean bacterial isolate, expect one to a handful of contigs and a genome size close to what is
expected for the species. Dozens of contigs usually mean too few or too short reads. Check the Kraken2
report if a large fraction of the reads was assigned to another organism.

Once you are satisfied with the results, `data/nextflow_work` can be deleted.
